<h1><center> Macromolecular Surface concentration </center></h1>

<h4><center> Author: Anisbel León Marcos${^1}$ </center></h4>
<h6><center> ${^1}$Institute for Tropospheric Research (TROPOS)
 ${^1}$leon@tropos.de  
</center></h6>
<h5><center> Contact information: </center></h5>
<h5><center> heinold@tropos.de (Bernd Heinold) </center></h5>
<br/>

### Models :

#### - Finite-Volume sea-ice ocean model  <a href="https://fesom.de/models/fesom20/">(FESOM2) </a>

#### - Regulated Ecosystem Model  <a href="https://ui.adsabs.harvard.edu/abs/2018PrOce.168...65S/abstract">(FESOM-REcoM2) </a>
> ##### Includes two phytoplankton classes and is coupled with FESOM2


### Equations: 
> #### Adapted from <a href="https://doi.org/10.5194/gmd-16-4883-2023">(Gürses et al. 2023) </a>

<br/>

##### Import packges

In [12]:
import numpy as np
import xarray as xr
import math
import glob

##### Set data directory path


In [2]:
data_dir = './'

## Variables for calculating ocean biomolecule concentration


#### Reading in FESOM-REcoM2 files 

In [5]:
# Reading files
def read_files(name):
    direct = glob.glob(f"{data_dir}{name}_*")
    direct.sort()
    return direct

dir_pcho = read_files('PCHO')
dir_diac = read_files('DiaC')
dir_dian = read_files('DiaN')
dir_phyc = read_files('PhyC')
dir_phyn = read_files('PhyN')


30

### Computing biomolecule surface ocean concentration

#### Polar lipids


In [14]:
def set_attrib_create_ncfile(conc, new_var_na, long_na):
    """Function to define netcdf attributes and create netcdf file with the biomolecule concentration"""
    conc = conc.rename(name_dict={'VAR':new_var_na})
    conc[new_var_na].attrs = dict(long_name=long_na,
                                  units = 'mmol C m-3',
                                  description=f'{new_var_na} ocean surface concentration in carbon units, {conc.time.dt.year.values[0]}')
    conc.attrs['title'] = f'{new_var_na} ocean surface concentration'
    
    yr = conc.time.dt.year.values[0] 
    new_file = f'./orig_data/{new_var_na}_var_regular_grid_interp_wv_res025_{yr}.nc'
    conc.to_netcdf(path=new_file)


In [9]:
tao_lip = 8     # [d]    lifetime of lipids in ocean surface water
                 #(Hopkinson et al. 2002, very labile DOC pool)
e_C_phy = 0.1    # [d−1]  Small phytoplankton excretion constant of organic C 
e_C_dia = 0.1    # [d−1]  Small diatom excretion constant of organic C

# limiter factor that downregulates the metabolic processes such as excretion of phytoplankton when 
# the intracellular nitrogen quota (qN:C) becomes too high
# f_lim(omega, q1, q2) = 1 - exp(-o_N_max (|del_q| - del_q)^2)
# del_q = q1 - q2
# f_lim(o_N_max, q1, q2)
# f_lim_phy(o_N_max, q_NC_phy, q_NC_phy_max) 
# f_lim_dia(o_N_max, q_NC_dia, q_NC_dia_max) 

o_N_max = 1000     # [mmolC mmolN-1] Maximum limiter regulater for N  
q_NC_phy_max = 0.2 # [mmolN mmolC-1] Maximum intracellular N : C ratio for small phytoplankton
q_NC_dia_max = 0.2 # [mmolN mmolC-1] Maximum intracellular N : C ratio for diatoms


for i,d in enumerate(dir_diac):
    #Reading files
    C_diaC =  xr.open_dataset(d)
    C_diaN =  xr.open_dataset(dir_dian[i])
    C_phyC =  xr.open_dataset(dir_phyc[i])
    C_phyN =  xr.open_dataset(dir_phyn[i])
    
    # calculating N:C ratio
    q_NC_phy = C_phyN/C_phyC
    q_NC_dia = C_diaN/C_diaC

    #calculation del_q(∆q)
    del_q_phy = q_NC_phy - q_NC_phy_max
    del_q_dia = q_NC_dia - q_NC_dia_max

    # calculating limiter factor    
    # flim(θ, q1, q2) = 1 − exp(−θ(|∆q| − ∆q)2)
    # ∆q = q1 −q2
    f_lim_phy = 1 - np.exp(-o_N_max * (np.abs(del_q_phy) - del_q_phy)** 2)  # f_lim_phy(o_N_max, q_NC_phy, q_NC_phy_max)     
    f_lim_dia = 1 - np.exp(-o_N_max * (np.abs(del_q_dia) - del_q_dia)** 2)  # f_lim_dia(o_N_max, q_NC_dia, q_NC_dia_max)  

    fpcho = 0.634   # fraction of PCHO of excreted organic carbon
    perct = 0.136612 # 5% of DOC_phy_ex

    # lipids concentration 
    C_li = perct * tao_lip * ((1-fpcho) * e_C_phy * f_lim_phy * C_phyC  + 
                                 (1-fpcho)* e_C_dia * f_lim_dia * C_diaC) # lipids concentration 

    # setting attributes
    C_li = set_attrib_create_ncfile(C_li, 'Lipids', 'Polar Lipids')


### Dissolved combined amino acids 


In [3]:
### DCAA is PCHO dependent
Ratio = 0.33  #(std=0.08)

In [11]:
for i,d in enumerate(dir_pcho):
    C_po= xr.open_dataset(d)
    C_po = set_attrib_create_ncfile(C_po, 'PCHO', 'Acidic Polysaccharides')
    C_pr = C_po*Ratio 
    
    # setting attributes
    C_pr = C_pr.rename(name_dict={'PCHO':'VAR'})
    C_pr = set_attrib_create_ncfile(C_pr, 'DAA', 'Dissolved combined amino acids')
